# Offline evaluation on `shop.history_sessions`

Before putting a new recommender live, estimate offline whether it beats the baseline.

The log was produced by the **old system** (best sellers plus a bit of randomness), so it only tells you what happened with the products *it* showed. Two estimates:

1. **Hit rate**: of the impressions where the shopper clicked, how often is the clicked product in your top 10? Easy, but biased towards popular products (the old system showed them most).
2. **Inverse propensity scoring (IPS)** of the click rate in **slot 1**: keep only the impressions where your first product equals the one the old system put in slot 1, and weight each by `1 / propensity`. This corrects the popularity bias of the log for that slot. SNIPS (self-normalised IPS) has lower variance.

We hold out the **last 7 days** of the history: the policies are built only from data before the cut and evaluated after it.

In [2]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q):
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

## Data

In [3]:
last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

# Columns are Dynamic (schemaless): filters work as is; joins and some aggregates need casts like ::String
test = sql(f"""
    SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
    FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
test["clicked"] = test.clicked.replace("", None)
print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))
test.head(3)

train <  2026-11-07 <= test
100000 test impressions, observed CTR 0.0371


,request_id,ts,user_id,context,recent_views,shown,propensity,clicked,purchased
0,r_14320eee98a40657,2026-11-13 01:07:32.855000000,u_4dc902f1,"{'device': 'android', 'page': 'product', 'regi...","[{'product_id': 'p_37e2ad', 'ts': '2026-11-13 ...","[p_e2454f, p_dc1c2e, p_0de352, p_3f8b7e, p_f18...","[0.299512, 0.112651, 0.029955, 0.00311, 0.0234...",p_e2454f,NaN
1,r_29f746d2fe7724fb,2026-11-11 04:49:19.295000000,u_d5ab45ef,"{'device': 'web', 'page': 'home', 'region': 'P...",[],"[p_8ca8ed, p_692114, p_2c88d5, p_4ba423, p_49d...","[0.218886, 0.019778, 0.289526, 0.027926, 0.018...",NaN,NaN
2,r_2e786b6eff9e23e1,2026-11-08 18:20:12.514000000,u_525e4c4c,"{'device': 'web', 'page': 'category', 'region'...","[{'product_id': 'p_07b81b', 'ts': '2026-11-08 ...","[p_e95fee, p_b62677, p_09a938, p_42d718, p_14b...","[0.011493, 0.000975, 0.114442, 0.001033, 0.001...",NaN,NaN


In [4]:
import sys
sys.path.insert(0, ".")
from recommender import best_sellers, recommend   # the same code that runs live

best = best_sellers(until=CUT)

In [ ]:
# Co-views computed inside RawTree: for every product, the 20 products most viewed in the same sessions
coviews = sql(f"""
    SELECT a.product_id AS p, b.product_id AS q, count() AS n
    FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS a
    INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS b
      ON a.session_id = b.session_id
    WHERE a.product_id != b.product_id
    GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p""")
covisit = coviews.groupby("p").q.apply(list).to_dict()

print(len(covisit), "products with co-views")
print("\n" + covisit)


5000 products with co-views
{'p_001897': ['p_9a06f9', 'p_6dac3f', 'p_6db845', 'p_86bf8d', 'p_982e1a', 'p_630d7a', 'p_b2413c', 'p_468cbf', 'p_ac3612', 'p_c928cc', 'p_52b8ee', 'p_4ba7e5', 'p_f110eb', 'p_7c47a2', 'p_91d47e', 'p_f40f39', 'p_7972bd', 'p_704fd9', 'p_9ae52f', 'p_e91877'], 'p_00196b': ['p_ba0231', 'p_9b9c09', 'p_8e1870', 'p_cc8e68', 'p_8d7325', 'p_fa4245', 'p_3a1db9', 'p_2fe246', 'p_94e11a', 'p_687a10', 'p_a8e89c', 'p_5c5e7e', 'p_368106', 'p_8e2a14', 'p_214275', 'p_093ee7', 'p_569398', 'p_b18603', 'p_cec9f4', 'p_6adc69'], 'p_001a20': ['p_2c88d5', 'p_cd5a6f', 'p_95b91e', 'p_f6ef7d', 'p_3bde83', 'p_051c87', 'p_c5f676', 'p_7a1c91', 'p_0fd01e', 'p_d49ad0', 'p_4077a5', 'p_882cdb', 'p_d75243', 'p_d94611', 'p_95d5df', 'p_49d2c3', 'p_6a33f2', 'p_819e00', 'p_b8227f', 'p_73ff68'], 'p_001d7f': ['p_aa3b8d', 'p_b69360', 'p_52e7e7', 'p_505e43', 'p_469130', 'p_bc49a5', 'p_022696', 'p_7f1865', 'p_8f64c9', 'p_52cb2a', 'p_26a29d', 'p_e19196', 'p_a350ae', 'p_92e7d7', 'p_a00067', 'p_e223d1', 'p_6

## Policies

A policy maps one request (a row) to an ordered list of 10 product IDs.

In [6]:
def baseline(row):
    """The live baseline, imported from starter/recommender.py."""
    return recommend(row.recent_views, best)

def coview(row):
    """Products co-viewed with this session's views (most recent first), then the baseline."""
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views)):
        for rank, q in enumerate(covisit.get(v["product_id"], [])):
            score[q] += 0.7 ** k / (rank + 1)
    items = [q for q, _ in score.most_common(10)]
    return (items + [p for p in baseline(row) if p not in items])[:10]

POLICIES = {"baseline": baseline, "coview": coview}

## Evaluation

In [7]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    w = match / df.propensity.str[0].to_numpy()
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

old system, observed slot-1 CTR: 0.0109


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.406519,0.007036,0.047872,3948.0
coview,0.245959,0.010984,0.014890,5294.0


## Next steps

- Few `slot1_matches` means a noisy IPS estimate: report it next to the estimate.
- Personal taste: who bought what in `shop.history_events` (matrix factorisation, item-to-item similarity).
- Cold start: new products only have a `description`. Text embeddings can place them next to similar products.
- Context: does price sensitivity change with `device` and `region`?
- Live: the stream changes (trends, price drops, stock-outs). Your `team_NN.feedback` tells you how you are really doing.